In [1]:
from dataGeneration import run_scr_model
from minMaxLHSampler import maximin_lhs

import numpy as np

mat_path = "./data/NEDC_lite.mat"

In [2]:
mean_k_ads = 4.32
mean_k_des = 4.83e04
mean_k_std = 3.59e09
mean_k_fst = 8.80e09
mean_k_slw = 3.56e11
mean_k_oxi = 6.99e05

one_std_k_ads = 1.8
one_std_k_des = 3.22e04
one_std_k_std = 7.18e07
one_std_k_fst = 1.76e08
one_std_k_slw = 7.13e09
one_std_k_oxi = 2.33e05

std_k_ads = mean_k_ads - one_std_k_ads
std_k_des = mean_k_des - one_std_k_des
std_k_std = mean_k_std - one_std_k_std
std_k_fst = mean_k_fst - one_std_k_fst
std_k_slw = mean_k_slw - one_std_k_slw
std_k_oxi = mean_k_oxi - one_std_k_oxi

In [3]:
up_k_ads = mean_k_ads + std_k_ads
up_k_des = mean_k_des + std_k_des
up_k_std = mean_k_std + std_k_std
up_k_fst = mean_k_fst + std_k_fst
up_k_slw = mean_k_slw + std_k_slw
up_k_oxi = mean_k_oxi + std_k_oxi

low_k_ads = mean_k_ads - std_k_ads
low_k_des = mean_k_des - std_k_des
low_k_std = mean_k_std - std_k_std
low_k_fst = mean_k_fst - std_k_fst
low_k_slw = mean_k_slw - std_k_slw
low_k_oxi = mean_k_oxi - std_k_oxi

lower_bounds = [low_k_ads, low_k_des, low_k_std, low_k_fst, low_k_slw, low_k_oxi]
upper_bounds = [up_k_ads, up_k_des, up_k_std, up_k_fst, up_k_slw, up_k_oxi]

# ads = 0
# des = 1
# std = 2
# fst = 3
# slw = 4
# oxi = 5


In [4]:
best_sample = maximin_lhs(lower_bounds=lower_bounds, upper_bounds=upper_bounds, n_samples=1000, n_candidates=10000)

In [5]:
np.save("lhs_sampling.npy", best_sample)
print(best_sample.shape)

(1000, 6)


In [6]:
from tqdm import tqdm

for _ in tqdm(range(len(best_sample))):
    k_ads = best_sample[_][0]
    k_des = best_sample[_][1]
    k_std = best_sample[_][2]
    k_fst = best_sample[_][3]
    k_slw = best_sample[_][4]
    k_oxi = best_sample[_][5]

    time_s, Model_sensor_ppm = run_scr_model(
                                mat_path = mat_path,
                                k_std_0=k_std,
                                k_fst_0=k_fst,
                                k_slw_0=k_slw,
                                k_ads_0=k_ads,
                                k_des_0=k_des,
                                k_nh3ox_0=k_oxi,
                            )
    
    if _ == 0:
        data = np.expand_dims(Model_sensor_ppm, axis = 0)
    else:
        data = np.concatenate((data, np.expand_dims(Model_sensor_ppm, axis = 0)))

    if ((_+1) % 100 == 0) and (_ != 0):
        print(f"Saved data for {_+1} samples.")
        np.save("lhs_data.npy", data)
    
np.save("lhs_data.npy", data)

 10%|█         | 100/1000 [33:18<4:56:10, 19.75s/it]

Saved data for 100 samples.


 20%|██        | 200/1000 [1:06:29<4:33:31, 20.51s/it]

Saved data for 200 samples.


 30%|███       | 300/1000 [1:39:22<3:47:31, 19.50s/it]

...

 90%|████████▉ | 899/1000 [5:04:13<34:01, 20.21s/it]  

Saved data for 900 samples.


100%|█████████▉| 999/1000 [5:37:04<00:20, 20.28s/it]

Saved data for 1000 samples.


100%|██████████| 1000/1000 [5:37:23<00:00, 20.24s/it]


## Splitting data into Train, Val, Test

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split

outputs = np.load("../Data/lhs_data.npy")
ks = np.load("../Data/lhs_sampling.npy")

outputs_train, outputs_test, ks_train, ks_test = train_test_split(outputs, ks, test_size=0.2, random_state=0) # 80% train, 20% val+test
outputs_val, outputs_test, ks_val, ks_test = train_test_split(outputs_test, ks_test, test_size=0.5, random_state=0) # 10% val, 10% test 

print(outputs_train.shape, outputs_val.shape, outputs_test.shape)
print(ks_train.shape, ks_val.shape, ks_test.shape)


(800, 11993) (100, 11993) (100, 11993)
(800, 6) (100, 6) (100, 6)


In [2]:
np.save("../Data/outputs_train.npy", outputs_train)
np.save("../Data/outputs_val.npy", outputs_val)
np.save("../Data/outputs_test.npy", outputs_test)
np.save("../Data/ks_train.npy", ks_train)
np.save("../Data/ks_val.npy", ks_val)
np.save("../Data/ks_test.npy", ks_test)